# Project: QA on Private Document


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [2]:
pip uninstall -y pinecone-client

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -U pinecone

  Using cached pinecone-10.0.0-cp310-abi3-win_amd64.whl.metadata (7.1 kB)
Using cached pinecone-10.0.0-cp310-abi3-win_amd64.whl (2.9 MB)
  Attempting uninstall: pinecone
    Found existing installation: pinecone 7.3.0
    Uninstalling pinecone-7.3.0:
      Successfully uninstalled pinecone-7.3.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-pinecone 0.2.13 requires pinecone[asyncio]<8.0.0,>=6.0.0, but you have pinecone 10.0.0 which is incompatible.


In [4]:
pip install pypdf -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install docx2text -q

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement docx2text (from versions: none)
ERROR: No matching distribution found for docx2text


In [6]:
pip install wikepedia -q

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement wikepedia (from versions: none)
ERROR: No matching distribution found for wikepedia


In [7]:
pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install -U langchain-community pypdf

Note: you may need to restart the kernel to use updated packages.


In [9]:
def load_document(file):

    import os

    name, ext = os.path.splitext(file)
    ext = ext.lower()

    if ext == '.pdf':
        from langchain_community.document_loaders import PyPDFLoader
        print(f"Loading {file}")
        loader = PyPDFLoader(file)

    elif ext == '.docx':
        from langchain_community.document_loaders import Docx2txtLoader
        print(f"Loading {file}")
        loader = Docx2txtLoader(file)

    else:
        print('Document format is not supported')
        return None

    data = loader.load()

    return data

def load_from_wikipedia(query, lang='en', load_max_docs=2):
    from langchain_community.document_loaders import WikipediaLoader

    loader = WikipediaLoader(
        query=query,
        lang=lang,
        load_max_docs=load_max_docs
    )

    data = loader.load()
    return data
    
def chunk_data(data, chunk_size=256):
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=0
    )

    chunks = text_splitter.split_documents(data)

    return chunks

def estimate_embedding_cost(chunks, price_per_million_tokens):
    # Rough estimate: 1 token ≈ 4 characters for English text
    total_chars = sum(len(chunk.page_content) for chunk in chunks)
    estimated_tokens = total_chars / 4

    cost = (estimated_tokens / 1_000_000) * price_per_million_tokens

    print(f"Chunks: {len(chunks):,}")
    print(f"Characters: {total_chars:,}")
    print(f"Estimated tokens: {estimated_tokens:,.0f}")
    print(f"Estimated cost: ${cost:.6f}")

    return cost

# Embedding and uploading to a vector database (pinecone)


In [10]:
pip install pinecone -q

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install -U langchain-pinecone pinecone langchain-openai

  Using cached pinecone-7.3.0-py3-none-any.whl.metadata (9.5 kB)
Using cached pinecone-7.3.0-py3-none-any.whl (587 kB)
  Attempting uninstall: pinecone
    Found existing installation: pinecone 10.0.0
    Uninstalling pinecone-10.0.0:
      Successfully uninstalled pinecone-10.0.0
Note: you may need to restart the kernel to use updated packages.


In [12]:
def insert_or_fetch_embeddings(index_name, chunks):
    from pinecone import Pinecone, ServerlessSpec
    from langchain_pinecone import PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings

    pc = Pinecone()

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        dimensions=1536
    )

    if index_name in pc.list_indexes().names():
        print(f"Index already exists: {index_name}")

        vector_store = PineconeVectorStore.from_existing_index(
            index_name=index_name,
            embedding=embeddings
        )

        print("OK")
        return vector_store

    print(f"Creating index: {index_name}")

    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    vector_store = PineconeVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        index_name=index_name
    )

    print("OK")
    return vector_store

In [13]:
def delete_pinecone_index_new(index_name="all"):
    from pinecone import Pinecone

    pc = Pinecone()

    if index_name == "all":
        indexes = pc.list_indexes().names()

        print("Deleting all indexes...")
        for index in indexes:
            print(f"Deleting: {index}")
            pc.delete_index(index)

        print("OK")

    else:
        print(f"Deleting index: {index_name}")
        pc.delete_index(index_name)
        print("OK")

# Running Code

In [14]:
data= load_document('files/rag_practice_guide.pdf')
#print(data[3].metadata)

print(f'you have {len(data)} pages in your data')
print(f'There are {len(data[2].page_content)} chars')

C:\Users\Rinku\AppData\Local\Temp\ipykernel_2108\994730198.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading files/rag_practice_guide.pdf
you have 5 pages in your data
There are 513 chars


In [15]:
data= load_document('files/Angular.docx')
#print(data[3].metadata)

#print(f'you have {len(data)} pages in your data')
#print(f'There are {len(data[0].page_content)} chars')
#print(data[0].page_content)

Loading files/Angular.docx


In [16]:
#data= load_from_wikipedia('GPT-4')
#print(data[0].page_content)

In [17]:
chunks=chunk_data(data)
#print(len(chunks))
#print(chunks[10].page_content)

estimate_embedding_cost(chunks, price_per_million_tokens=0.02)

Chunks: 20
Characters: 4,270
Estimated tokens: 1,068
Estimated cost: $0.000021


2.135e-05

In [18]:
delete_pinecone_index_new()

Deleting all indexes...
Deleting: askadocument
OK


In [19]:
index_name='askadocument'
vector_store=insert_or_fetch_embeddings(index_name,chunks)

Creating index: askadocument
OK


# Asking and Getting Asnwer

In [22]:
def ask_and_get_answer(vector_store, q):
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=1
    )

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    )

    documents = retriever.invoke(q)

    context = "\n\n".join(
        doc.page_content for doc in documents
    )

    prompt = ChatPromptTemplate.from_template(
        """Answer the question based only on the following context.

Context:
{context}

Question:
{question}

Answer:"""
    )

    chain = prompt | llm | StrOutputParser()

    return chain.invoke({
        "context": context,
        "question": q
    })

In [25]:
q = "binding "
answer = ask_and_get_answer(vector_store, q)
print(answer)

Binding in Angular refers to the process of synchronizing data between the components and the view. There are different types of binding:

1. **Interpolation (`{{}}`)**: This is a one-way data binding method where the data flows from the component's TypeScript code to the template (view). It's primarily used for displaying dynamic content.

2. **Property Binding**: This is also a one-way data binding but it allows you to dynamically set values in the HTML elements' properties.

3. **Event Binding**: This is a one-way binding where the data flow is from the view to the component. It uses parentheses `()` and is typically used to listen for events such as clicks, keypresses, etc.

4. **Two-Way Binding**: This method, depicted by `[()]`, allows for data to be synchronized both ways - from the component to the view and from the view to the component. This is often used with forms where user input needs to be captured and processed in the component.


In [29]:
import time
i=1
print('write Quit or Exist to quit')

while True:
    q=input(f'question #{1}: ')
    i=i+1
    if( q.lower() in ['quit', 'exist']):
        time.sleep(2)
        break

    answer=ask_and_get_answer(vector_store, q)
    print(f'\nAnswer: {answer}')
    print(f'\n {"-"  * 50}\n')
        

write Quit or Exist to quit


question #1:  waht is anguler?



Answer: Angular is a platform and framework for building single-page client applications using HTML and TypeScript. It is developed and maintained by Google and provides a comprehensive ecosystem for building dynamic web applications with a focus on performance and modularity. Angular includes built-in tools and features such as templates, dependency injection, end-to-end tooling, and integrated best practices to address developers' common challenges in creating robust applications.

 --------------------------------------------------



question #1:  binding



Answer: Interpolation: {{}} is one way

Property Binding: two way

Event Binding: One way with ()

Two Way Binding: [()]

 --------------------------------------------------



...

question #1:  Write syntax for @Input and @Output properties5



Answer: ```typescript
@Input() inpute1: string = "";

@Output() testEvent = new EventEmitter<string>();
```

 --------------------------------------------------



question #1:  quite



Answer: The provided context does not contain any information related to the term "quite." Please provide additional or different context for a more accurate response.

 --------------------------------------------------



question #1:  exist


In [31]:
delete_pinecone_index_new()

Deleting all indexes...
Deleting: askadocument
OK


In [33]:
data= load_from_wikipedia('chatgpt','en')
chucks=chunk_data(data)
index_name='chatgpt'
vector_store=insert_or_fetch_embenddings(index_name)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# using Chroma as a Vector DB

In [34]:
pip install -q chromadb

Note: you may need to restart the kernel to use updated packages.


In [42]:
pip install -U langchain-chroma chromadb

Note: you may need to restart the kernel to use updated packages.


In [43]:
def create_embedding_chroma(chunks, persistent_directory="./chroma.db"):
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        dimensions=1536
    )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persistent_directory
    )

    return vector_store


def load_embedding_chroma(persistent_directory="./chroma.db"):
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        dimensions=1536
    )

    vector_store = Chroma(
        persist_directory=persistent_directory,
        embedding_function=embeddings
    )

    return vector_store

In [44]:
data=load_document('files/EAadhaar_Sankirna_updated.pdf')
chucks=chunk_data(data, chunk_size=100)
vector_store=create_embedding_chroma(chucks)

Loading files/EAadhaar_Sankirna_updated.pdf


In [46]:
q='what is aadhar number?'
anwer=ask_and_get_answer(vector_store,q)
print(anwer)

4940 6663 5845


In [47]:
db=load_embedding_chroma()
q='give me address'
anwer=ask_and_get_answer(vector_store,q)
print(anwer)

C-115 Kamdhenu Woods, Block No 2, Ankhol Gam, Ankhol, PO: Ankhol, VTC: Ankhol, District: Vadodara, State: Gujarat, PIN Code: 390019.


In [54]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=1
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

chat_history = []

def ask_question(q):
    documents = retriever.invoke(q)

    context = "\n\n".join(
        doc.page_content for doc in documents
    )

    history = "\n".join(
        f"User: {msg.content}" if isinstance(msg, HumanMessage)
        else f"Assistant: {msg.content}"
        for msg in chat_history
    )

    prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using the
provided context and conversation history.

Conversation history:
{history}

Context:
{context}

Question:
{question}

Answer:
""")

    chain = prompt | llm

    response = chain.invoke({
        "history": history,
        "context": context,
        "question": q
    })

    chat_history.append(HumanMessage(content=q))
    chat_history.append(AIMessage(content=response.content))

    return response

In [50]:
data=load_document('files/EAadhaar_Sankirna_updated.pdf')
chucks=chunk_data(data, chunk_size=100)
vector_store=create_embedding_chroma(chucks)

Loading files/EAadhaar_Sankirna_updated.pdf


In [55]:
q ='what is aadhar card?'
result=ask_question(q)

In [56]:
result

AIMessage(content="An Aadhaar card is a unique identification card issued by the Government of India to each individual resident of India. It contains a 12-digit unique identification number, known as the Aadhaar number, which is linked to the individual's biometric and demographic data. The Aadhaar card serves as proof of identity and address anywhere in India. It is commonly used for various purposes, such as opening a bank account, registering for government services, and verifying identity.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 234, 'total_tokens': 323, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06

In [61]:
q ='what is  mobile number?'
result=ask_question(q)

In [62]:
chat_history

[HumanMessage(content='what is aadhar card?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="An Aadhaar card is a unique identification card issued by the Government of India to each individual resident of India. It contains a 12-digit unique identification number, known as the Aadhaar number, which is linked to the individual's biometric and demographic data. The Aadhaar card serves as proof of identity and address anywhere in India. It is commonly used for various purposes, such as opening a bank account, registering for government services, and verifying identity.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is  mobile number?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The mobile number is 9558813659.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
# using a 